### Initial Database and  Checks

In [8]:
import os
import numpy as np
import tensorflow as tf
from datasets import load_dataset
from sklearn.model_selection import train_test_split
# 1. Clear Ghost Memory: 
# Previous crashes leave "zombie" graphs in the GPU memory. This flushes them out.
tf.keras.backend.clear_session()

# 2. Enable Memory Growth:
# Prevents TensorFlow from blindly allocating 100% of the VRAM immediately.
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    try:
        for gpu in physical_devices:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

# 3. Enable Mixed Precision (The Silver Bullet):
# This forces the GPU to use float16 for calculations but keeps variables in float32. 
# It literally cuts your VRAM requirement in half without hurting accuracy.
tf.keras.mixed_precision.set_global_policy('mixed_float16')

os.environ['TF_CPP_MIN_LOG_LEVEL'] = "3"

# === GPU CHECK ===
print("=" * 60)
print("GPU/CUDA Configuration Check")
print("=" * 60)
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Devices Available: {len(physical_devices)}")
if physical_devices:
    for i, gpu in enumerate(physical_devices):
        print(f"  GPU {i}: {gpu}")
else:
    print("  ⚠️  WARNING: No GPUs detected! Training will be SLOW.")

# Display nvidia-smi output
print("\n" + "=" * 60)
print("NVIDIA GPU Status (nvidia-smi)")
print("=" * 60)
try:
    import subprocess
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=5)
    print(result.stdout)
    if result.stderr:
        print("Errors:", result.stderr)
except Exception as e:
    print(f"Could not run nvidia-smi: {e}")
print("=" * 60 + "\n")

# NOTE: may have to change batch size and epochs depending on GPU VRAM. but epochs should be kept to 100 if possible for train accuracy.
BATCH_SIZE = 32
EPOCHS = 100
LEARNING_RATE = 1e-3
SAMPLE_SIZE = 15000

# Load Dataset
dataset = load_dataset("zh-plus/tiny-imagenet")
train_cover, train_secret = train_test_split(dataset['train']['image'][:SAMPLE_SIZE], train_size=0.5, shuffle=True)
test_cover, test_secret = train_test_split(dataset['valid']['image'][:1000], train_size=0.5, shuffle=True)
holdout_cover, holdout_secret = train_test_split(
    dataset['train']['image'][SAMPLE_SIZE:SAMPLE_SIZE + 1000], # Taking the next 1000 images
    train_size=0.5, 
    shuffle=True
)
del dataset # Free memory

STEPS = len(train_cover) // BATCH_SIZE + 1

# === OPTIMIZATION: Pre-convert all PIL images to numpy arrays once ===
# This eliminates the ~5-10ms PIL conversion overhead on EVERY training step
print("Converting images to numpy arrays (one-time conversion)...")
train_cover_np = np.array([np.array(img.convert('RGB'), dtype=np.float32) for img in train_cover])
train_secret_np = np.array([np.array(img.convert('RGB'), dtype=np.float32) for img in train_secret])
test_cover_np = np.array([np.array(img.convert('RGB'), dtype=np.float32) for img in test_cover])
test_secret_np = np.array([np.array(img.convert('RGB'), dtype=np.float32) for img in test_secret])
holdout_cover_np = np.array([np.array(img.convert('RGB'), dtype=np.float32) for img in holdout_cover])
holdout_secret_np = np.array([np.array(img.convert('RGB'), dtype=np.float32) for img in holdout_secret])
print(f"✓ Image conversion complete! ({len(train_cover_np)} training pairs)")

# Build optimized tf.data pipelines
# Switched from from_generator() to from_tensor_slices() for 2-3x faster data loading
# Increased shuffle buffer from 1000 → 10000 for better shuffling
train_dataset = tf.data.Dataset.from_tensor_slices(
    (train_cover_np, train_secret_np)
).shuffle(10000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices(
    (test_cover_np, test_secret_np)
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

holdout_dataset = tf.data.Dataset.from_tensor_slices(
    (holdout_cover_np, holdout_secret_np)
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

GPU/CUDA Configuration Check
TensorFlow version: 2.21.0
GPU Devices Available: 1
  GPU 0: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

NVIDIA GPU Status (nvidia-smi)
Sun Mar 22 00:28:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.54                 Driver Version: 595.79         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 Ti     On  |   00000000:01:00.0  On |                  N/A |
|  0%   41C    P8              8W /  165W |  

### NN Arch and Setup

In [9]:
@tf.function
def conv2d(input, filters, biases, strides=1):
    x = tf.nn.conv2d(input, filters, strides=[1, strides, strides, 1], padding='SAME')
    x = tf.nn.bias_add(x, biases)
    return tf.nn.relu(x)

@tf.function
def conv2d_sigmoid(input, filters, biases, strides=1):
    x = tf.nn.conv2d(input, filters, strides=[1, strides, strides, 1], padding='SAME')
    x = tf.nn.bias_add(x, biases)
    return tf.nn.sigmoid(x) # Forces output to be between 0.0 and 1.0

initializer = tf.initializers.glorot_normal(12541)

In [10]:
WEIGHTS_PREP_NETWORK = {
    'conv3x3_1': tf.Variable(initializer([3, 3, 3, 50])),
    'conv3x3_2': tf.Variable(initializer([3, 3, 50, 50])),
    'conv3x3_3': tf.Variable(initializer([3, 3, 50, 50])),
    'conv3x3_4': tf.Variable(initializer([3, 3, 50, 50])),

    'conv4x4_1': tf.Variable(initializer([4, 4, 3, 50])),
    'conv4x4_2': tf.Variable(initializer([4, 4, 50, 50])),
    'conv4x4_3': tf.Variable(initializer([4, 4, 50, 50])),
    'conv4x4_4': tf.Variable(initializer([4, 4, 50, 50])),

    'conv5x5_1': tf.Variable(initializer([5, 5, 3, 50])),
    'conv5x5_2': tf.Variable(initializer([5, 5, 50, 50])),
    'conv5x5_3': tf.Variable(initializer([5, 5, 50, 50])),
    'conv5x5_4': tf.Variable(initializer([5, 5, 50, 50])),

    'conv3x3_5': tf.Variable(initializer([3, 3, 150, 50])),
    'conv4x4_5': tf.Variable(initializer([4, 4, 150, 50])),
    'conv5x5_5': tf.Variable(initializer([5, 5, 150, 50]))
}

BIASES_PREP_NETWORK = {
    'conv3x3_1': tf.Variable(tf.zeros([50])),
    'conv3x3_2': tf.Variable(tf.zeros([50])),
    'conv3x3_3': tf.Variable(tf.zeros([50])),
    'conv3x3_4': tf.Variable(tf.zeros([50])),

    'conv4x4_1': tf.Variable(tf.zeros([50])),
    'conv4x4_2': tf.Variable(tf.zeros([50])),
    'conv4x4_3': tf.Variable(tf.zeros([50])),
    'conv4x4_4': tf.Variable(tf.zeros([50])),

    'conv5x5_1': tf.Variable(tf.zeros([50])),
    'conv5x5_2': tf.Variable(tf.zeros([50])),
    'conv5x5_3': tf.Variable(tf.zeros([50])),
    'conv5x5_4': tf.Variable(tf.zeros([50])),


    'conv3x3_5': tf.Variable(tf.zeros([50])),
    'conv4x4_5': tf.Variable(tf.zeros([50])),
    'conv5x5_5': tf.Variable(tf.zeros([50]))
}

@tf.function
def prep_network(secret_tensor):

    secret_tensor = tf.reshape(secret_tensor, [-1, 64, 64, 3])
    secret_tensor = tf.cast(secret_tensor, tf.float32)
    secret_tensor = tf.math.divide(secret_tensor, 255.)

    with tf.name_scope('prep_network'):
        with tf.name_scope('conv3x3'):
            conv3x3 = secret_tensor
            conv3x3 = conv2d(conv3x3, WEIGHTS_PREP_NETWORK['conv3x3_1'], BIASES_PREP_NETWORK['conv3x3_1'])
            conv3x3 = conv2d(conv3x3, WEIGHTS_PREP_NETWORK['conv3x3_2'], BIASES_PREP_NETWORK['conv3x3_2'])
            conv3x3 = conv2d(conv3x3, WEIGHTS_PREP_NETWORK['conv3x3_3'], BIASES_PREP_NETWORK['conv3x3_3'])
            conv3x3 = conv2d(conv3x3, WEIGHTS_PREP_NETWORK['conv3x3_4'], BIASES_PREP_NETWORK['conv3x3_4'])

        with tf.name_scope('conv4x4'):
            conv4x4 = secret_tensor
            conv4x4 = conv2d(conv4x4, WEIGHTS_PREP_NETWORK['conv4x4_1'], BIASES_PREP_NETWORK['conv4x4_1'])
            conv4x4 = conv2d(conv4x4, WEIGHTS_PREP_NETWORK['conv4x4_2'], BIASES_PREP_NETWORK['conv4x4_2'])
            conv4x4 = conv2d(conv4x4, WEIGHTS_PREP_NETWORK['conv4x4_3'], BIASES_PREP_NETWORK['conv4x4_3'])
            conv4x4 = conv2d(conv4x4, WEIGHTS_PREP_NETWORK['conv4x4_4'], BIASES_PREP_NETWORK['conv4x4_4'])

        with tf.name_scope('conv5x5'):
            conv5x5 = secret_tensor
            conv5x5 = conv2d(conv5x5, WEIGHTS_PREP_NETWORK['conv5x5_1'], BIASES_PREP_NETWORK['conv5x5_1'])
            conv5x5 = conv2d(conv5x5, WEIGHTS_PREP_NETWORK['conv5x5_2'], BIASES_PREP_NETWORK['conv5x5_2'])
            conv5x5 = conv2d(conv5x5, WEIGHTS_PREP_NETWORK['conv5x5_3'], BIASES_PREP_NETWORK['conv5x5_3'])
            conv5x5 = conv2d(conv5x5, WEIGHTS_PREP_NETWORK['conv5x5_4'], BIASES_PREP_NETWORK['conv5x5_4'])

        concat_1 = tf.concat([conv3x3, conv4x4, conv5x5], axis=3)

        conv3x3 = conv2d(concat_1, WEIGHTS_PREP_NETWORK['conv3x3_5'], BIASES_PREP_NETWORK['conv3x3_5'])
        conv4x4 = conv2d(concat_1, WEIGHTS_PREP_NETWORK['conv4x4_5'], BIASES_PREP_NETWORK['conv4x4_5'])
        conv5x5 = conv2d(concat_1, WEIGHTS_PREP_NETWORK['conv5x5_5'], BIASES_PREP_NETWORK['conv5x5_5'])

        return tf.concat([conv3x3, conv4x4, conv5x5], axis=3)
    


In [11]:
WEIGHTS_HIDE_NETWORK = {
    'conv3x3_1': tf.Variable(initializer([3, 3, 153, 50])),
    'conv3x3_2': tf.Variable(initializer([3, 3, 50, 50])),
    'conv3x3_3': tf.Variable(initializer([3, 3, 50, 50])),
    'conv3x3_4': tf.Variable(initializer([3, 3, 50, 50])),

    'conv4x4_1': tf.Variable(initializer([4, 4, 153, 50])),
    'conv4x4_2': tf.Variable(initializer([4, 4, 50, 50])),
    'conv4x4_3': tf.Variable(initializer([4, 4, 50, 50])),
    'conv4x4_4': tf.Variable(initializer([4, 4, 50, 50])),

    'conv5x5_1': tf.Variable(initializer([5, 5, 153, 50])),
    'conv5x5_2': tf.Variable(initializer([5, 5, 50, 50])),
    'conv5x5_3': tf.Variable(initializer([5, 5, 50, 50])),
    'conv5x5_4': tf.Variable(initializer([5, 5, 50, 50])),

    'conv3x3_5': tf.Variable(initializer([3, 3, 150, 50])),
    'conv4x4_5': tf.Variable(initializer([4, 4, 150, 50])),
    'conv5x5_5': tf.Variable(initializer([5, 5, 150, 50])),

    'out': tf.Variable(initializer([1, 1, 150, 3])),
}

BIASES_HIDE_NETWORK = {
    'conv3x3_1': tf.Variable(tf.zeros([50])),
    'conv3x3_2': tf.Variable(tf.zeros([50])),
    'conv3x3_3': tf.Variable(tf.zeros([50])),
    'conv3x3_4': tf.Variable(tf.zeros([50])),

    'conv4x4_1': tf.Variable(tf.zeros([50])),
    'conv4x4_2': tf.Variable(tf.zeros([50])),
    'conv4x4_3': tf.Variable(tf.zeros([50])),
    'conv4x4_4': tf.Variable(tf.zeros([50])),

    'conv5x5_1': tf.Variable(tf.zeros([50])),
    'conv5x5_2': tf.Variable(tf.zeros([50])),
    'conv5x5_3': tf.Variable(tf.zeros([50])),
    'conv5x5_4': tf.Variable(tf.zeros([50])),


    'conv3x3_5': tf.Variable(tf.zeros([50])),
    'conv4x4_5': tf.Variable(tf.zeros([50])),
    'conv5x5_5': tf.Variable(tf.zeros([50])),

    'out': tf.Variable(tf.zeros([3]))
}

@tf.function
def hide_network(cover_tensor, prep_tensor):

    cover_tensor = tf.reshape(cover_tensor, [-1, 64, 64, 3])
    cover_tensor = tf.cast(cover_tensor, tf.float32)
    cover_tensor = tf.math.divide(cover_tensor, 255.)

    with tf.name_scope('hide_network'):
        concat_1 = tf.concat([cover_tensor, prep_tensor], axis=3)

        with tf.name_scope('conv3x3'):
            conv3x3 = concat_1
            conv3x3 = conv2d(conv3x3, WEIGHTS_HIDE_NETWORK['conv3x3_1'], BIASES_HIDE_NETWORK['conv3x3_1'])
            conv3x3 = conv2d(conv3x3, WEIGHTS_HIDE_NETWORK['conv3x3_2'], BIASES_HIDE_NETWORK['conv3x3_2'])
            conv3x3 = conv2d(conv3x3, WEIGHTS_HIDE_NETWORK['conv3x3_3'], BIASES_HIDE_NETWORK['conv3x3_3'])
            conv3x3 = conv2d(conv3x3, WEIGHTS_HIDE_NETWORK['conv3x3_4'], BIASES_HIDE_NETWORK['conv3x3_4'])

        with tf.name_scope('conv4x4'):
            conv4x4 = concat_1
            conv4x4 = conv2d(conv4x4, WEIGHTS_HIDE_NETWORK['conv4x4_1'], BIASES_HIDE_NETWORK['conv4x4_1'])
            conv4x4 = conv2d(conv4x4, WEIGHTS_HIDE_NETWORK['conv4x4_2'], BIASES_HIDE_NETWORK['conv4x4_2'])
            conv4x4 = conv2d(conv4x4, WEIGHTS_HIDE_NETWORK['conv4x4_3'], BIASES_HIDE_NETWORK['conv4x4_3'])
            conv4x4 = conv2d(conv4x4, WEIGHTS_HIDE_NETWORK['conv4x4_4'], BIASES_HIDE_NETWORK['conv4x4_4'])

        with tf.name_scope('conv5x5'):
            conv5x5 = concat_1
            conv5x5 = conv2d(conv5x5, WEIGHTS_HIDE_NETWORK['conv5x5_1'], BIASES_HIDE_NETWORK['conv5x5_1'])
            conv5x5 = conv2d(conv5x5, WEIGHTS_HIDE_NETWORK['conv5x5_2'], BIASES_HIDE_NETWORK['conv5x5_2'])
            conv5x5 = conv2d(conv5x5, WEIGHTS_HIDE_NETWORK['conv5x5_3'], BIASES_HIDE_NETWORK['conv5x5_3'])
            conv5x5 = conv2d(conv5x5, WEIGHTS_HIDE_NETWORK['conv5x5_4'], BIASES_HIDE_NETWORK['conv5x5_4'])

        concat_2 = tf.concat([conv3x3, conv4x4, conv5x5], axis=3)

        conv3x3 = conv2d(concat_2, WEIGHTS_HIDE_NETWORK['conv3x3_5'], BIASES_HIDE_NETWORK['conv3x3_5'])
        conv4x4 = conv2d(concat_2, WEIGHTS_HIDE_NETWORK['conv4x4_5'], BIASES_HIDE_NETWORK['conv4x4_5'])
        conv5x5 = conv2d(concat_2, WEIGHTS_HIDE_NETWORK['conv5x5_5'], BIASES_HIDE_NETWORK['conv5x5_5'])

        concat_final = tf.concat([conv3x3, conv4x4, conv5x5], axis=3)

        return conv2d_sigmoid(concat_final, WEIGHTS_HIDE_NETWORK['out'], BIASES_HIDE_NETWORK['out'])

In [12]:
WEIGHTS_REVEAL_NETWORK = {
    'conv3x3_1': tf.Variable(initializer([3, 3, 3, 50])),
    'conv3x3_2': tf.Variable(initializer([3, 3, 50, 50])),
    'conv3x3_3': tf.Variable(initializer([3, 3, 50, 50])),
    'conv3x3_4': tf.Variable(initializer([3, 3, 50, 50])),

    'conv4x4_1': tf.Variable(initializer([4, 4, 3, 50])),
    'conv4x4_2': tf.Variable(initializer([4, 4, 50, 50])),
    'conv4x4_3': tf.Variable(initializer([4, 4, 50, 50])),
    'conv4x4_4': tf.Variable(initializer([4, 4, 50, 50])),

    'conv5x5_1': tf.Variable(initializer([5, 5, 3, 50])),
    'conv5x5_2': tf.Variable(initializer([5, 5, 50, 50])),
    'conv5x5_3': tf.Variable(initializer([5, 5, 50, 50])),
    'conv5x5_4': tf.Variable(initializer([5, 5, 50, 50])),

    'conv3x3_5': tf.Variable(initializer([3, 3, 150, 50])),
    'conv4x4_5': tf.Variable(initializer([4, 4, 150, 50])),
    'conv5x5_5': tf.Variable(initializer([5, 5, 150, 50])),

    'out': tf.Variable(initializer([1, 1, 150, 3])),
}

BIASES_REVEAL_NETWORK = {
    'conv3x3_1': tf.Variable(tf.zeros([50])),
    'conv3x3_2': tf.Variable(tf.zeros([50])),
    'conv3x3_3': tf.Variable(tf.zeros([50])),
    'conv3x3_4': tf.Variable(tf.zeros([50])),

    'conv4x4_1': tf.Variable(tf.zeros([50])),
    'conv4x4_2': tf.Variable(tf.zeros([50])),
    'conv4x4_3': tf.Variable(tf.zeros([50])),
    'conv4x4_4': tf.Variable(tf.zeros([50])),

    'conv5x5_1': tf.Variable(tf.zeros([50])),
    'conv5x5_2': tf.Variable(tf.zeros([50])),
    'conv5x5_3': tf.Variable(tf.zeros([50])),
    'conv5x5_4': tf.Variable(tf.zeros([50])),


    'conv3x3_5': tf.Variable(tf.zeros([50])),
    'conv4x4_5': tf.Variable(tf.zeros([50])),
    'conv5x5_5': tf.Variable(tf.zeros([50])),

    'out': tf.Variable(tf.zeros([3]))
}

@tf.function
def reveal_network(hide_tensor):

    hide_tensor = tf.reshape(hide_tensor, [-1, 64, 64, 3])
    hide_tensor = tf.cast(hide_tensor, tf.float32)
    hide_tensor = tf.math.divide(hide_tensor, 255.)

    with tf.name_scope('reveal_network'):
        with tf.name_scope('conv3x3'):
            conv3x3 = hide_tensor
            conv3x3 = conv2d(conv3x3, WEIGHTS_REVEAL_NETWORK['conv3x3_1'], BIASES_REVEAL_NETWORK['conv3x3_1'])
            conv3x3 = conv2d(conv3x3, WEIGHTS_REVEAL_NETWORK['conv3x3_2'], BIASES_REVEAL_NETWORK['conv3x3_2'])
            conv3x3 = conv2d(conv3x3, WEIGHTS_REVEAL_NETWORK['conv3x3_3'], BIASES_REVEAL_NETWORK['conv3x3_3'])
            conv3x3 = conv2d(conv3x3, WEIGHTS_REVEAL_NETWORK['conv3x3_4'], BIASES_REVEAL_NETWORK['conv3x3_4'])

        with tf.name_scope('conv4x4'):
            conv4x4 = hide_tensor
            conv4x4 = conv2d(conv4x4, WEIGHTS_REVEAL_NETWORK['conv4x4_1'], BIASES_REVEAL_NETWORK['conv4x4_1'])
            conv4x4 = conv2d(conv4x4, WEIGHTS_REVEAL_NETWORK['conv4x4_2'], BIASES_REVEAL_NETWORK['conv4x4_2'])
            conv4x4 = conv2d(conv4x4, WEIGHTS_REVEAL_NETWORK['conv4x4_3'], BIASES_REVEAL_NETWORK['conv4x4_3'])
            conv4x4 = conv2d(conv4x4, WEIGHTS_REVEAL_NETWORK['conv4x4_4'], BIASES_REVEAL_NETWORK['conv4x4_4'])

        with tf.name_scope('conv5x5'):
            conv5x5 = hide_tensor
            conv5x5 = conv2d(conv5x5, WEIGHTS_REVEAL_NETWORK['conv5x5_1'], BIASES_REVEAL_NETWORK['conv5x5_1'])
            conv5x5 = conv2d(conv5x5, WEIGHTS_REVEAL_NETWORK['conv5x5_2'], BIASES_REVEAL_NETWORK['conv5x5_2'])
            conv5x5 = conv2d(conv5x5, WEIGHTS_REVEAL_NETWORK['conv5x5_3'], BIASES_REVEAL_NETWORK['conv5x5_3'])
            conv5x5 = conv2d(conv5x5, WEIGHTS_REVEAL_NETWORK['conv5x5_4'], BIASES_REVEAL_NETWORK['conv5x5_4'])

        concat_1 = tf.concat([conv3x3, conv4x4, conv5x5], axis=3)

        conv3x3 = conv2d(concat_1, WEIGHTS_REVEAL_NETWORK['conv3x3_5'], BIASES_REVEAL_NETWORK['conv3x3_5'])
        conv4x4 = conv2d(concat_1, WEIGHTS_REVEAL_NETWORK['conv4x4_5'], BIASES_REVEAL_NETWORK['conv4x4_5'])
        conv5x5 = conv2d(concat_1, WEIGHTS_REVEAL_NETWORK['conv5x5_5'], BIASES_REVEAL_NETWORK['conv5x5_5'])

        concat_final = tf.concat([conv3x3, conv4x4, conv5x5], axis=3)

        return conv2d_sigmoid(concat_final, WEIGHTS_REVEAL_NETWORK['out'], BIASES_REVEAL_NETWORK['out'])

### Loss func and Training Loop

In [ ]:
import time
from datetime import timedelta

@tf.function
def steganorgraphy_loss(cover_input, secret_input, cover_output, secret_output, beta=1.0):
    beta = tf.constant(beta, name="beta")

    cover_mse = tf.reduce_mean(tf.keras.losses.MSE(cover_input, cover_output))
    secret_mse = tf.reduce_mean(tf.keras.losses.MSE(secret_input, secret_output))

    total_loss = cover_mse + beta * secret_mse

    return total_loss, cover_mse, secret_mse

optimizer = tf.optimizers.Adam(LEARNING_RATE)

from IPython.display import clear_output

# Pre-compute trainable variables once (don't recreate every step)
trainable_variables = list(WEIGHTS_PREP_NETWORK.values()) + \
    list(WEIGHTS_HIDE_NETWORK.values()) + \
    list(WEIGHTS_REVEAL_NETWORK.values()) + \
    list(BIASES_PREP_NETWORK.values()) + \
    list(BIASES_HIDE_NETWORK.values()) + \
    list(BIASES_REVEAL_NETWORK.values())

start_time = time.time()
ema_epoch_time = None  # Exponential moving average for better time prediction
EMA_ALPHA = 0.3  # Weight recent epochs more heavily (0.3 = 30% new, 70% historical)
epoch_logs = []  # Persist epoch completion messages

# Helper function for formatting time as h:m:s (hides 0 units)
def format_time_hms(seconds):
    h = int(seconds) // 3600
    m = (int(seconds) % 3600) // 60
    s = int(seconds) % 60
    if h > 0:
        return f"{h}h:{m:02d}m:{s:02d}s"
    elif m > 0:
        return f"{m}m:{s:02d}s"
    else:
        return f"{s}s"

# Logging function for epoch completion - prints persistently
def log_epoch(epoch_num, elapsed, cover_loss_val, secret_loss_val, total_loss_val, ema_str):
    log_msg = f"\n🎯 Epoch {epoch_num}/{EPOCHS} | ⏱️  {elapsed} | Cover Loss: {cover_loss_val:.6f} | Secret Loss: {secret_loss_val:.6f} | Total Loss: {total_loss_val:.6f} | EMA: {ema_str}"
    epoch_logs.append(log_msg)
    print(log_msg)

# Training step: forward pass, loss calculation, and gradient update
def training_step(cover, secret, trainable_variables, optimizer):
    """Execute one training step: forward pass, loss calc, and backprop."""
    with tf.GradientTape() as tape:
        scaled_cover = tf.cast(cover, tf.float32) / 255.0
        scaled_secret = tf.cast(secret, tf.float32) / 255.0
        prep_output = prep_network(scaled_secret)
        hide_output = hide_network(scaled_cover, prep_output)
        reveal_output = reveal_network(hide_output)
        total_loss, cover_loss, secret_loss = steganorgraphy_loss(
            scaled_cover, scaled_secret, hide_output, reveal_output
        )
    
    grads = tape.gradient(total_loss, trainable_variables)
    optimizer.apply_gradients(zip(grads, trainable_variables))
    
    return total_loss, cover_loss, secret_loss

# Time and metrics calculations
def calculate_training_metrics(epoch, step, start_time, epoch_start, ema_epoch_time, elapsed_epoch):
    """Calculate time predictions and training progress metrics."""
    current_time = time.time()
    elapsed_total = current_time - start_time
    
    # Per-epoch stats
    steps_remaining_in_epoch = STEPS - (step + 1)
    time_per_step = elapsed_epoch / (step + 1)
    eta_epoch = steps_remaining_in_epoch * time_per_step
    
    # Overall training stats (using EMA for better prediction)
    avg_epoch_time = ema_epoch_time if ema_epoch_time is not None else elapsed_epoch
    epochs_remaining = EPOCHS - (epoch + 1)
    eta_total = epochs_remaining * avg_epoch_time + eta_epoch
    
    # Progress percentage
    total_steps_done = epoch * STEPS + (step + 1)
    total_steps = EPOCHS * STEPS
    progress_pct = (total_steps_done / total_steps) * EPOCHS
    
    return {
        'elapsed_total': elapsed_total,
        'eta_epoch': eta_epoch,
        'eta_total': eta_total,
        'time_per_step': time_per_step,
        'progress_pct': progress_pct
    }

# Display training step output
def display_step_output(epoch_num, progress_pct, step, total_loss_val, cover_loss_val, 
                       secret_loss_val, elapsed_epoch, elapsed_total, time_per_step, eta_epoch, eta_total):
    """Display formatted training progress and metrics."""
    elapsed_str = format_time_hms(elapsed_epoch)
    elapsed_total_str = format_time_hms(elapsed_total)
    eta_epoch_str = format_time_hms(eta_epoch)
    eta_total_str = format_time_hms(eta_total)
    
    clear_output(wait=True)
    # Reprint all epoch completion logs to persist them across clears
    for log_msg in epoch_logs:
        print(log_msg, end='')
    print(f"Epoch {epoch_num}/{EPOCHS} | Progress: {progress_pct:.1f}%")
    print(f"Step {step + 1:>4}/{STEPS} [{elapsed_str} current elapsed epoch < {eta_epoch_str} eta epoch left, {1/time_per_step:.2f}it/s]")
    print(f"  Cover loss:  {cover_loss_val:.6f}")
    print(f"  Secret loss: {secret_loss_val:.6f}")
    print(f"  Total loss:  {total_loss_val:.6f}")
    print(f"Samples processed: {(step + 1) * BATCH_SIZE}")
    print(f"\nTotal elapsed: {elapsed_total_str} | ETA: {eta_total_str} | Predicted total: {format_time_hms(elapsed_total + eta_total)}")


Epoch 1/100 | Progress: 0.0%
Step    1/235 [1s current elapsed epoch < 5m:03s eta epoch left, 0.77it/s]
  Cover loss:  0.066319
  Secret loss: 0.075676
  Total loss:  0.141995
Samples processed: 32

Total elapsed: 1s | ETA: 7m:11s | Predicted total: 7m:12s


KeyboardInterrupt: 

In [ ]:
for epoch in range(EPOCHS):
    epoch_start = time.time()
    for step, (cover, secret) in enumerate(train_dataset):
        step_start = time.time()
        total_loss, cover_loss, secret_loss = training_step(cover, secret, trainable_variables, optimizer)

        # Reduce output frequency - only print every 50 steps
        if (step % 50 == 0) or (step == STEPS - 1):
            elapsed_epoch = time.time() - epoch_start
            metrics = calculate_training_metrics(epoch, step, start_time, epoch_start, ema_epoch_time, elapsed_epoch)
            
            display_step_output(
                epoch + 1, metrics['progress_pct'], step,
                float(np.mean(total_loss)), float(np.mean(cover_loss)), float(np.mean(secret_loss)),
                elapsed_epoch, metrics['elapsed_total'], metrics['time_per_step'],
                metrics['eta_epoch'], metrics['eta_total']
            )

    # End of Epoch Summary
    epoch_duration = time.time() - epoch_start
    # Update exponential moving average for better future predictions
    if ema_epoch_time is None:
        ema_epoch_time = epoch_duration
    else:
        ema_epoch_time = EMA_ALPHA * epoch_duration + (1 - EMA_ALPHA) * ema_epoch_time
    epoch_duration_str = format_time_hms(epoch_duration)
    
    # Log epoch completion with losses
    log_epoch(
        epoch_num=epoch + 1,
        elapsed=epoch_duration_str,
        cover_loss_val=float(np.mean(cover_loss)),
        secret_loss_val=float(np.mean(secret_loss)),
        total_loss_val=float(np.mean(total_loss)),
        ema_str=format_time_hms(ema_epoch_time)
    )
clear_output(wait=True)
print("Final Results, All samples processed")
print(f"  Cover loss:  {np.mean(cover_loss):.6f}")
print(f"  Secret loss: {np.mean(secret_loss):.6f}")
print(f"  Total loss:  {np.mean(total_loss):.6f}")
total_duration = format_time_hms(time.time() - start_time)
print(f"\n>>>> Total Runtime: {total_duration}")

In [ ]:
loss_mean = list()

for step, (cover, secret) in enumerate(test_dataset):
    prep_output = prep_network(secret)
    hide_output = hide_network(cover, prep_output)
    reveal_output = reveal_network(hide_output)
    total_loss, cover_loss, secret_loss = steganorgraphy_loss(cover, secret, hide_output, reveal_output)

    loss_mean.append(total_loss)

    clear_output()
    print("Total loss: %.4f" % (np.mean(loss_mean)))

### Final Testing / Example Output

In [ ]:
import matplotlib.pyplot as plt

# Use pre-converted numpy arrays instead of PIL objects
cover_img_array = holdout_cover_np[0]
secret_img_array = holdout_secret_np[0]

# Run Inference
prep_output = prep_network(secret_img_array)
hide_output = hide_network(cover_img_array, prep_output)
reveal_output = reveal_network(hide_output)

# Post-process (Scale back to 0-255)
hide_result = np.clip(hide_output.numpy().squeeze() * 255, 0, 255).astype(np.uint8)
reveal_result = np.clip(reveal_output.numpy().squeeze() * 255, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
images = [
    np.clip(secret_img_array.squeeze() * 255, 0, 255).astype(np.uint8),
    np.clip(cover_img_array.squeeze() * 255, 0, 255).astype(np.uint8),
    hide_result, 
    reveal_result
]
titles = ["Original Secret", "Original Cover", "Cover with Secret", "Secret Revealed"]

for i, ax in enumerate(axes):
    # Ensure images are displayed correctly even if they are in float format
    ax.imshow(images[i])
    ax.set_title(titles[i])
    ax.axis('off') # Hide the X/Y pixel coordinates for a cleaner look

plt.tight_layout()
plt.show()